In [ ]:
import torch
import torch.nn as nn
import time
from torch.nn import functional as F
import os
import glob
import csv
import matplotlib.pyplot as plt

# to use kaggle
# mkdir -p ~/.kaggle
# download from kaggle.json from google drive and send to ~/.kaggle
from kaggle.api.kaggle_api_extended import KaggleApi
kaggle_api = KaggleApi()
kaggle_api.authenticate()

device = 'cuda' if torch.cuda.is_available() else 'cpu'

#HYPERPARAMS
block_size = 256
batch_size = 128
max_iterations = 50000
learning_rate = 3e-4
eval_iters = 50
n_embd = 256 #total number of dimensions to be captured
n_head = 8 # how many heads are running in parallel
n_layer = 12 # no. of decoder blocks
dropout = 0.3
checkpoint_interval = 4000

print(device)

cpu


In [ ]:
from datasets import load_dataset

# to use kaggle
# mkdir -p ~/.kaggle
# download from kaggle.json from google drive and send to ~/.kaggle
datasets_root = "Datasets"
os.makedirs(datasets_root, exist_ok=True)

allowed_chars = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?;:()[]{}'\"-\n")
text = ""

kaggle_dataset_path = os.path.join(datasets_root, "final-fantasy-dialogue-scripts")
os.makedirs(kaggle_dataset_path, exist_ok=True)
kaggle_api.dataset_download_files(
    "tylerhuxtable/final-fantasy-dialogue-scripts",
    path=kaggle_dataset_path,
    unzip=True
    )

# IF MORE KAGGLE DATASETS TO BE ADDED,
#   kaggle_api.dataset_download_files(
#       "THE DATASET TO BE LOADED",
#       path=kaggle_dataset_path,
#       unzip=True
#   )

csv_files = glob.glob(os.path.join(kaggle_dataset_path, "*.csv"))
for file in csv_files:
    with open(file, 'r', encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            if len(row) >0:
                text+= row[0] + "\n"

obt_processed_path = os.path.join(datasets_root, "openwebtext_cleaned.txt")

if os.path.exists(obt_processed_path):
    print("Existing OBT Dataset found. Loading.")
    with open(obt_processed_path, "r", encoding="utf-8") as f:
        text = f.read()
else:
    print("Cleaned OBT Dataset was not found, making new.")

    openwebtext_dataset = load_dataset(
        "openwebtext",
        streaming=True,
        split="train",
        trust_remote_code=True
    )

    for i, sample in enumerate(openwebtext_dataset):
        text += sample["text"] + "\n"
        if i > 50000:
            break

    text = "".join(c for c in text if c in allowed_chars)

    with open(obt_processed_path, "w", encoding="utf-8") as f:
        f.write(text)
    
chars =  sorted(set(text))
vocab_size=len(chars)

KeyboardInterrupt: 

In [ ]:
#mapping from strings to integers
string_to_integer = {ch:i for i,ch in enumerate(chars) }
#mapping from integers to strings
integer_to_string = {i:ch for i, ch in enumerate(chars) }

encode = lambda s: [string_to_integer[c] for c in s]
decode = lambda l: ''.join([integer_to_string[i] for i in l])

# example: encoding "Hello"
# print(encode('Hello'))
# H -> 31, e -> 57, l -> 64, o -> 67

# example: decoding "[31 57 64 64 67]"
# print(decode([31, 57, 64, 64, 67]))
# 31->H, 57->e, 64->l, 67->o

data = torch.tensor(encode(text), dtype=torch.long)
# print(data[:100])

tensor([41, 57, 54,  0, 41, 67, 50, 56, 54, 53, 58, 54,  0, 64, 55,  0, 34, 50,
        52, 51, 54, 69, 57, 22, 52, 69, 70, 68,  0, 37, 67, 58, 62, 70, 68,  8,
         0, 40, 52, 64, 54, 63, 50,  0, 37, 67, 58, 62, 50,  8, 41, 57, 70, 63,
        53, 54, 67,  0, 50, 63, 53,  0, 33, 58, 56, 57, 69, 63, 58, 63, 56,  8,
         0, 26, 63, 69, 54, 67,  0, 69, 57, 67, 54, 54,  0, 44, 58, 69, 52, 57,
        54, 68,  8,  0,  0, 10,  8,  0, 44, 57])


In [ ]:
n = int(0.8*len(data))
#80-20 split, 80% is train, testing is 20%
train_data = data[:n]
val_data = data[n:]
batch_size = int(batch_size)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device),y.to(device)
    return x,y
    
x,y= get_batch('train')
# print('inputs:')
# print(x)
# print('targets:')
# print(y)

In [5]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([41]) target is tensor(57)
when input is tensor([41, 57]) target is tensor(54)
when input is tensor([41, 57, 54]) target is tensor(0)
when input is tensor([41, 57, 54,  0]) target is tensor(41)
when input is tensor([41, 57, 54,  0, 41]) target is tensor(67)
when input is tensor([41, 57, 54,  0, 41, 67]) target is tensor(50)
when input is tensor([41, 57, 54,  0, 41, 67, 50]) target is tensor(56)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56]) target is tensor(54)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56, 54]) target is tensor(53)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56, 54, 53]) target is tensor(58)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56, 54, 53, 58]) target is tensor(54)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56, 54, 53, 58, 54]) target is tensor(0)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56, 54, 53, 58, 54,  0]) target is tensor(64)
when input is tensor([41, 57, 54,  0, 41, 67, 50, 56, 54, 53, 58, 

In [6]:
def get_random_chunk(split):
    """
    Returns a random chunk of the dataset as a torch tensor.
    
    split: 'train' or 'val'
    """
    data_split = train_data if split == 'train' else val_data
    start_idx = torch.randint(0, len(data_split) - block_size, (1,)).item()
    chunk = data_split[start_idx : start_idx + block_size]
    return chunk.to(device)

In [7]:
@torch.no_grad()
def loss_estimate():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    #updates weights and biases
    return out

In [8]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        #n_embed is dim(embedding)
        #n_head is number of heads
        head_size = n_embd // n_head
        # head_size is number of features each head will be capturing in multi head attention
        self.sa = MultiHeadAttention(n_head, head_size)
        # sa = SelfAttention
        self.ffwd = FeedForward(n_embd)
        # Feed Forward : Linear -> ReLU -> Linear
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        # these are just for prenorm and postnorm respectively.

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x+y)
        y = self.ffwd(x)
        x = self.ln2(x+y)
        #self attention -> add a norm -> feed forward -> add a norm
        
        return x

In [9]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential (
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self,x):
        return self.net(x)

In [10]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size*num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [11]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril' , torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # computing attention scores
        wei= q @ k.transpose(-2,-1)*k.shape[-1]**-0.5 
        # (B, t, head_size) @ (B, head_size, T) -> (B, T, T) 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #(B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)

        # performing weighted aggregation of values
        v = self.value(x) #(B, T, head_size)
        out = torch.matmul(wei, v)
        # (B,T,T) @ (B, T, head_size) -> (B, T, head_size)
        return out


In [12]:
class GPTLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) #norm of final layer
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            
    def forward(self, index, targets=None):
        B,T = index.shape
        token_emb = self.token_embedding_table(index)
        posn_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = token_emb + posn_emb 
        x = self.blocks(x)
        x= self.ln_f(x)
        logits = self.lm_head(x)
        if targets==None:
            loss = None
        else:
            # batches, time, channels
            B, T, C = logits.shape
            #.view helps us to make a tensor with those dimensions
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is a (B,T) array of indices in the present context
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            # retrive the predictions
            logits, loss = self.forward(index_cond)
            # focus on the last time-step
            logits = logits[:,-1,:] # (B,C)
            # apply softmax on logits to get the probabilities
            probs = F.softmax(logits, dim= -1) # (B,C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B,1)
            index = torch.cat((index, index_next), dim=1) # (B,T+1)
        return index

model = GPTLM(vocab_size)
m = model.to(device)
# torch.long is just int64
context = torch.zeros((1,1), dtype=torch.long, device=device)

print('loading model params...')
model = GPTLM(vocab_size).to(device)  # fresh instance
model.load_state_dict(torch.load('model-01.pth'))
model.train()
print('loaded successfully.')
# gen_char = decode(m.generate(context, max_new_tokens=500)[0].tolist())
# print(gen_char)

loading model params...
loaded successfully.


In [13]:
def format_time(seconds):
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hrs:02d}:{mins:02d}:{secs:02d}"

In [87]:
#creating a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_losses_history = []
val_losses_history = []
iterations_history = []
best_val_loss = float('inf')
best_model_path = None

tic = time.time()
last_eval_time = tic
cumulative_time = 0.0


for i in range(max_iterations):
    if i%eval_iters == 0:
        current_time = time.time()
        losses = loss_estimate()
        train_losses_history.append(losses['train'])
        val_losses_history.append(losses['val'])
        iterations_history.append(i)
        
        if i == 0:
            print(f'Step:{i}, training loss: {losses["train"]:.3f}, val loss: {losses["val"]:.3f}')
        else:
            step_duration = current_time - last_eval_time
            cumulative_time += step_duration
            print(f'Step:{i}, training loss: {losses["train"]:.3f}, val loss: {losses["val"]:.3f}')
            print(f'time for last {eval_iters} iterations: {step_duration:.1f}s, Total time till now: {format_time(cumulative_time)}\n')
            
        if losses['val'] < best_val_loss:
            if best_model_path is not None and os.path.exists(best_model_path):
                os.remove(best_model_path)
            best_val_loss = losses['val']
            best_model_path = os.path.join("model_checkpoints", f'best_model_step_{i}.pth')
            torch.save(model.state_dict(), best_model_path)
            print(f'*** New best model saved at step {i} with val loss {best_val_loss:.3f} ***')
            
        last_eval_time = current_time
    # sampling a batch of data
    xb, yb = get_batch('train')

    #computing loss
    logits, loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if i > 0 and i % checkpoint_interval == 0:
        checkpoint_path = os.path.join("model_checkpoints", f'model-step-{i}.pth')
        torch.save(model.state_dict(), checkpoint_path)
        print(f'Checkpoint saved at step {i}: {checkpoint_path}')
toc = time.time()
print(loss.item())
total_time = toc - tic
print(f'total time for {max_iterations} iterations: {format_time(total_time)}')
    
print("\n Loss Plot: ")
plt.figure(figsize=(10, 6))
plt.plot(iterations_history, train_losses_history, label='Training Loss', marker='o', linestyle='--')
plt.plot(iterations_history, val_losses_history, label='Validation Loss', marker='x', linestyle='-')
plt.title('train loss and val loss over Iterations')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

if best_model_path is not None:
    best_model_final_path = 'model-01.pth'
    shutil.copy(best_model_path, best_model_final_path)
    print(f'Best model copied to {best_model_final_path}')

Step:0, training loss: 2.201, val loss: 2.186
Step:50, training loss: 2.170, val loss: 2.201
time for last 50 iterations: 27.9s, Total time till now: 00:00:27

Step:100, training loss: 2.147, val loss: 2.142
time for last 50 iterations: 27.0s, Total time till now: 00:00:54

Step:150, training loss: 2.190, val loss: 2.165
time for last 50 iterations: 27.1s, Total time till now: 00:01:21

Step:200, training loss: 2.111, val loss: 2.145
time for last 50 iterations: 26.9s, Total time till now: 00:01:48

Step:250, training loss: 2.107, val loss: 2.135
time for last 50 iterations: 27.5s, Total time till now: 00:02:16

Step:300, training loss: 2.132, val loss: 2.132
time for last 50 iterations: 27.6s, Total time till now: 00:02:43

Step:350, training loss: 2.104, val loss: 2.133
time for last 50 iterations: 27.3s, Total time till now: 00:03:11

Step:400, training loss: 2.091, val loss: 2.121
time for last 50 iterations: 27.1s, Total time till now: 00:03:38

Step:450, training loss: 2.104, val

KeyboardInterrupt: 